In [9]:
#Spark connection with S3 options
import os
import socket
from pyspark.sql import SparkSession

# Замените следующие значения на свои credentials
aws_access_key = "j612yzOjcIwYyPhVD14P"
aws_secret_key = "bVTEwJ7sJ0TboubM5wJcn2ieYrV91uXk3N9dJ457"
s3_bucket = "startde-datasets"
s3_endpoint_url = "https://s3.lab.karpov.courses"
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = """/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar, 
/nfs/env/lib/python3.8/site-packages/pyspark/jars/hadoop-aws-3.3.4.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/aws-java-sdk-bundle-1.12.433.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/postgresql-42.7.4.jar
"""

MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        config("spark.hadoop.fs.s3a.access.key", aws_access_key). \
        config("spark.hadoop.fs.s3a.secret.key", aws_secret_key). \
        config("fs.s3a.endpoint", s3_endpoint_url).  \
        config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem"). \
        config("spark.hadoop.fs.s3a.committer.name", "directory"). \
        config("spark.hadoop.fs.s3a.path.style.access", True). \
        config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
        config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false"). \
        getOrCreate()

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Read products table
products_table = spark.read.parquet("s3a://startde-datasets/shared/products.parquet")

In [11]:
products_table.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         0|   product_0|   22|
|         1|   product_1|   30|
|         2|   product_2|   91|
|         3|   product_3|   37|
|         4|   product_4|  145|
|         5|   product_5|  128|
|         6|   product_6|   66|
|         7|   product_7|  145|
|         8|   product_8|   51|
|         9|   product_9|   44|
|        10|  product_10|   53|
|        11|  product_11|   13|
|        12|  product_12|  104|
|        13|  product_13|  102|
|        14|  product_14|   24|
|        15|  product_15|   14|
|        16|  product_16|   38|
|        17|  product_17|   72|
|        18|  product_18|   16|
|        19|  product_19|   46|
+----------+------------+-----+
only showing top 20 rows



In [12]:
products_table = products_table.withColumn("name_suffix", substring(col("product_name"), -6, 6))
products_table.show()

+----------+------------+-----+-----------+
|product_id|product_name|price|name_suffix|
+----------+------------+-----+-----------+
|         0|   product_0|   22|     duct_0|
|         1|   product_1|   30|     duct_1|
|         2|   product_2|   91|     duct_2|
|         3|   product_3|   37|     duct_3|
|         4|   product_4|  145|     duct_4|
|         5|   product_5|  128|     duct_5|
|         6|   product_6|   66|     duct_6|
|         7|   product_7|  145|     duct_7|
|         8|   product_8|   51|     duct_8|
|         9|   product_9|   44|     duct_9|
|        10|  product_10|   53|     uct_10|
|        11|  product_11|   13|     uct_11|
|        12|  product_12|  104|     uct_12|
|        13|  product_13|  102|     uct_13|
|        14|  product_14|   24|     uct_14|
|        15|  product_15|   14|     uct_15|
|        16|  product_16|   38|     uct_16|
|        17|  product_17|   72|     uct_17|
|        18|  product_18|   16|     uct_18|
|        19|  product_19|   46| 

In [13]:
products_table = products_table.groupBy('name_suffix').agg(avg('price').alias('avg_price'))
products_table.show()
products_table.count()

+-----------+---------+
|name_suffix|avg_price|
+-----------+---------+
|     ct_373|     31.0|
|     ct_398|     94.0|
|     ct_480|     72.0|
|     ct_761|     48.0|
|     ct_803|    104.0|
|     ct_987|     73.0|
|     t_1420|    121.0|
|     t_1469|     62.0|
|     t_1850|     88.0|
|     t_2782|     59.0|
|     t_2918|     57.0|
|     t_3166|    108.0|
|     t_3339|     81.0|
|     t_3374|     65.0|
|     t_3525|     70.0|
|     t_3663|     43.0|
|     t_3706|     32.0|
|     t_3734|    141.0|
|     t_3821|     55.0|
|     t_4095|     66.0|
+-----------+---------+
only showing top 20 rows



75000

In [14]:
products_table = products_table.orderBy('name_suffix')
products_table.show()

+-----------+---------+
|name_suffix|avg_price|
+-----------+---------+
|     _10000|    143.0|
|     _10001|    149.0|
|     _10002|     90.0|
|     _10003|    137.0|
|     _10004|     89.0|
|     _10005|     14.0|
|     _10006|    147.0|
|     _10007|     75.0|
|     _10008|    134.0|
|     _10009|     50.0|
|     _10010|    101.0|
|     _10011|    120.0|
|     _10012|      7.0|
|     _10013|     86.0|
|     _10014|     13.0|
|     _10015|    106.0|
|     _10016|     38.0|
|     _10017|     50.0|
|     _10018|     34.0|
|     _10019|     57.0|
+-----------+---------+
only showing top 20 rows



In [16]:
result = products_table
pandas_df = result.toPandas()
pandas_df.to_parquet('result.parquet')

In [17]:
spark.stop()